In [3]:
# ============================================================
# CELL 1 — SETUP AND CONFIGURATION
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# ------------------------------------------------------------
# File configuration
# ------------------------------------------------------------

V11_DATASET_FILE = "viirs_v11_final_event_dataset.csv"

QC_OUTPUT_DIRECTORY = "dataset_qc"

os.makedirs(QC_OUTPUT_DIRECTORY, exist_ok=True)

# ------------------------------------------------------------
# Expected dataset configuration
# ------------------------------------------------------------

EXPECTED_EVENT_COUNT = 4893
EXPECTED_FEATURE_COUNT = 24

# ------------------------------------------------------------
# General geographic validity limits
# ------------------------------------------------------------

MIN_VALID_LATITUDE = -90.0
MAX_VALID_LATITUDE = 90.0

MIN_VALID_LONGITUDE = -180.0
MAX_VALID_LONGITUDE = 180.0

print("Dataset QC environment initialized.")
print("Input dataset :", V11_DATASET_FILE)
print("Output folder :", QC_OUTPUT_DIRECTORY)

Dataset QC environment initialized.
Input dataset : viirs_v11_final_event_dataset.csv
Output folder : dataset_qc


In [4]:
# ============================================================
# CELL 2 — LOAD V11 DATASET
# ============================================================

if not os.path.exists(V11_DATASET_FILE):
    raise FileNotFoundError(
        f"Dataset not found: {V11_DATASET_FILE}"
    )

v11_events = pd.read_csv(
    V11_DATASET_FILE,
    parse_dates=["start_date", "end_date"]
)

print("V11 dataset loaded successfully.")
print()
print("Shape:", v11_events.shape)
print("Rows :", len(v11_events))
print("Cols :", len(v11_events.columns))

V11 dataset loaded successfully.

Shape: (4893, 24)
Rows : 4893
Cols : 24


In [5]:
# ============================================================
# CELL 3 — DATASET STRUCTURE
# ============================================================

print("DATASET COLUMNS")
print("=" * 60)

for column_number, column_name in enumerate(
    v11_events.columns,
    start=1
):
    print(f"{column_number:2d}. {column_name}")

print("\nDATA TYPES")
print("=" * 60)

display(
    v11_events.dtypes
    .to_frame(name="data_type")
)

print("\nFIRST 5 EVENTS")
print("=" * 60)

display(v11_events.head())

DATASET COLUMNS
 1. event_id
 2. mean_frp
 3. max_frp
 4. std_frp
 5. mean_bright_ti4
 6. max_bright_ti4
 7. std_bright_ti4
 8. mean_bright_ti5
 9. max_bright_ti5
10. std_bright_ti5
11. start_date
12. end_date
13. active_days
14. detection_count
15. duration_days
16. activity_frequency
17. detections_per_active_day
18. centroid_lat
19. centroid_lon
20. daily_object_count
21. spatial_diameter_km
22. frp_range
23. ti4_range
24. ti5_range

DATA TYPES


,data_type
event_id,int64
mean_frp,float64
max_frp,float64
std_frp,float64
mean_bright_ti4,float64
max_bright_ti4,float64
std_bright_ti4,float64
mean_bright_ti5,float64
max_bright_ti5,float64
std_bright_ti5,float64



FIRST 5 EVENTS


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,detections_per_active_day,centroid_lat,centroid_lon,daily_object_count,spatial_diameter_km,frp_range,ti4_range,ti5_range
0,0,1.6354,4.9700,0.9465,307.7935,337.4800,10.9815,284.2007,289.9600,3.5646,2024-01-01,2024-01-31,27,85,31,0.8710,3.1481,23.1695,82.3411,75,1.5451,3.3346,29.6865,5.7593
1,1,1.4333,2.3200,0.7805,302.4200,306.6300,4.8742,286.5167,286.9300,0.3630,2024-01-01,2024-01-02,2,3,2,1.0000,1.5000,24.2093,82.7125,2,0.3679,0.8867,4.2100,0.4133
2,2,1.7355,3.9000,0.7277,309.4428,330.2000,8.5311,289.7826,296.0800,2.6531,2024-01-01,2024-01-31,27,85,31,0.8710,3.1481,22.0539,88.1241,64,1.0219,2.1645,20.7572,6.2974
3,3,1.4367,2.0700,0.5637,302.4333,307.3100,4.8066,285.6167,286.5000,0.7848,2024-01-01,2024-01-02,2,3,2,1.0000,1.5000,24.2042,82.7115,2,0.3496,0.6333,4.8767,0.8833
4,4,0.8000,0.8000,0.0000,302.0950,302.9700,1.2374,288.4150,288.5200,0.1485,2024-01-01,2024-01-01,1,2,1,1.0000,2.0000,22.3196,82.5665,1,0.3579,0.0000,0.8750,0.1050


In [6]:
# ============================================================
# CELL 4 — MISSING VALUE AUDIT
# ============================================================

missing_value_report = pd.DataFrame({
    "column": v11_events.columns,
    "missing_count": v11_events.isna().sum().values,
    "missing_percentage": (
        v11_events.isna().mean().values * 100
    )
})

missing_value_report = (
    missing_value_report
    .sort_values(
        "missing_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("MISSING VALUE REPORT")
print("=" * 60)

display(
    missing_value_report.round(4)
)

total_missing_values = v11_events.isna().sum().sum()

print("\nTotal missing values:", total_missing_values)

if total_missing_values == 0:
    print("STATUS: PASS — No missing values detected.")
else:
    print("STATUS: WARNING — Missing values detected.")

MISSING VALUE REPORT


,column,missing_count,missing_percentage
0,event_id,0,0.0000
1,mean_frp,0,0.0000
2,max_frp,0,0.0000
3,std_frp,0,0.0000
4,mean_bright_ti4,0,0.0000
5,max_bright_ti4,0,0.0000
6,std_bright_ti4,0,0.0000
7,mean_bright_ti5,0,0.0000
8,max_bright_ti5,0,0.0000
9,std_bright_ti5,0,0.0000



Total missing values: 0
STATUS: PASS — No missing values detected.


In [7]:
# ============================================================
# CELL 5 — DUPLICATE AND EVENT-ID INTEGRITY
# ============================================================

total_rows = len(v11_events)

unique_event_ids = v11_events["event_id"].nunique()

duplicate_event_id_count = (
    v11_events["event_id"]
    .duplicated()
    .sum()
)

duplicate_row_count = (
    v11_events
    .duplicated()
    .sum()
)

missing_event_id_count = (
    v11_events["event_id"]
    .isna()
    .sum()
)

event_id_integrity_report = pd.DataFrame({
    "check": [
        "Total rows",
        "Unique event IDs",
        "Duplicate event IDs",
        "Missing event IDs",
        "Completely duplicated rows"
    ],
    "value": [
        total_rows,
        unique_event_ids,
        duplicate_event_id_count,
        missing_event_id_count,
        duplicate_row_count
    ]
})

print("EVENT ID INTEGRITY")
print("=" * 60)

display(event_id_integrity_report)

if (
    unique_event_ids == total_rows
    and duplicate_event_id_count == 0
    and missing_event_id_count == 0
    and duplicate_row_count == 0
):
    print("STATUS: PASS — Event IDs are unique and complete.")
else:
    print("STATUS: WARNING — Event ID integrity issue detected.")

EVENT ID INTEGRITY


,check,value
0,Total rows,4893
1,Unique event IDs,4893
2,Duplicate event IDs,0
3,Missing event IDs,0
4,Completely duplicated rows,0


STATUS: PASS — Event IDs are unique and complete.


In [8]:
# ============================================================
# CELL 6 — DATA TYPE VALIDATION
# ============================================================

expected_numeric_columns = [
    "event_id",
    "mean_frp",
    "max_frp",
    "std_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "active_days",
    "detection_count",
    "duration_days",
    "activity_frequency",
    "detections_per_active_day",
    "centroid_lat",
    "centroid_lon",
    "daily_object_count",
    "spatial_diameter_km",
    "frp_range",
    "ti4_range",
    "ti5_range"
]

expected_datetime_columns = [
    "start_date",
    "end_date"
]

missing_expected_columns = [
    column
    for column in (
        expected_numeric_columns +
        expected_datetime_columns
    )
    if column not in v11_events.columns
]

print("Missing expected columns:")
print(missing_expected_columns)

numeric_type_report = pd.DataFrame({
    "column": expected_numeric_columns,
    "is_numeric": [
        pd.api.types.is_numeric_dtype(
            v11_events[column]
        )
        if column in v11_events.columns
        else False
        for column in expected_numeric_columns
    ]
})

datetime_type_report = pd.DataFrame({
    "column": expected_datetime_columns,
    "is_datetime": [
        pd.api.types.is_datetime64_any_dtype(
            v11_events[column]
        )
        if column in v11_events.columns
        else False
        for column in expected_datetime_columns
    ]
})

print("\nNUMERIC COLUMN CHECK")
display(numeric_type_report)

print("\nDATETIME COLUMN CHECK")
display(datetime_type_report)

Missing expected columns:
[]

NUMERIC COLUMN CHECK


,column,is_numeric
0,event_id,True
1,mean_frp,True
2,max_frp,True
3,std_frp,True
4,mean_bright_ti4,True
5,max_bright_ti4,True
6,std_bright_ti4,True
7,mean_bright_ti5,True
8,max_bright_ti5,True
9,std_bright_ti5,True



DATETIME COLUMN CHECK


,column,is_datetime
0,start_date,True
1,end_date,True


In [9]:
# ============================================================
# CELL 7 — COORDINATE AND DATE VALIDITY
# ============================================================

# ------------------------------------------------------------
# Coordinate checks
# ------------------------------------------------------------

invalid_latitude_mask = (
    (v11_events["centroid_lat"] < MIN_VALID_LATITUDE) |
    (v11_events["centroid_lat"] > MAX_VALID_LATITUDE)
)

invalid_longitude_mask = (
    (v11_events["centroid_lon"] < MIN_VALID_LONGITUDE) |
    (v11_events["centroid_lon"] > MAX_VALID_LONGITUDE)
)

invalid_latitude_count = invalid_latitude_mask.sum()
invalid_longitude_count = invalid_longitude_mask.sum()

# ------------------------------------------------------------
# Date checks
# ------------------------------------------------------------

invalid_start_date_count = (
    v11_events["start_date"].isna().sum()
)

invalid_end_date_count = (
    v11_events["end_date"].isna().sum()
)

end_before_start_mask = (
    v11_events["end_date"] <
    v11_events["start_date"]
)

end_before_start_count = end_before_start_mask.sum()

# ------------------------------------------------------------
# Dataset temporal range
# ------------------------------------------------------------

dataset_start_date = v11_events["start_date"].min()
dataset_end_date = v11_events["end_date"].max()

coordinate_date_report = pd.DataFrame({
    "check": [
        "Invalid latitude",
        "Invalid longitude",
        "Missing start date",
        "Missing end date",
        "End date before start date"
    ],
    "count": [
        invalid_latitude_count,
        invalid_longitude_count,
        invalid_start_date_count,
        invalid_end_date_count,
        end_before_start_count
    ]
})

print("COORDINATE AND DATE VALIDITY")
print("=" * 60)

display(coordinate_date_report)

print("\nCoordinate range:")
print(
    f"Latitude : {v11_events['centroid_lat'].min():.6f}"
    f" to {v11_events['centroid_lat'].max():.6f}"
)

print(
    f"Longitude: {v11_events['centroid_lon'].min():.6f}"
    f" to {v11_events['centroid_lon'].max():.6f}"
)

print("\nEvent date range:")
print(dataset_start_date)
print("to")
print(dataset_end_date)

COORDINATE AND DATE VALIDITY


,check,count
0,Invalid latitude,0
1,Invalid longitude,0
2,Missing start date,0
3,Missing end date,0
4,End date before start date,0



Coordinate range:
Latitude : 8.243390 to 34.630080
Longitude: 68.575770 to 97.075460

Event date range:
2024-01-01 00:00:00
to
2024-01-31 00:00:00


In [10]:
# ============================================================
# CELL 8 — THERMAL FEATURE QC
# ============================================================

thermal_feature_columns = [
    "mean_frp",
    "max_frp",
    "std_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "frp_range",
    "ti4_range",
    "ti5_range"
]

thermal_qc_records = []

for column in thermal_feature_columns:

    column_values = v11_events[column]

    thermal_qc_records.append({
        "feature": column,
        "minimum": column_values.min(),
        "maximum": column_values.max(),
        "mean": column_values.mean(),
        "median": column_values.median(),
        "negative_count": (
            column_values < 0
        ).sum(),
        "zero_count": (
            column_values == 0
        ).sum(),
        "infinite_count": (
            np.isinf(column_values)
        ).sum()
    })

thermal_qc_report = pd.DataFrame(
    thermal_qc_records
)

print("THERMAL FEATURE QC")
print("=" * 60)

display(
    thermal_qc_report.round(4)
)

# ------------------------------------------------------------
# Basic thermal consistency checks
# ------------------------------------------------------------

frp_relationship_violations = (
    v11_events["max_frp"] <
    v11_events["mean_frp"]
).sum()

ti4_relationship_violations = (
    v11_events["max_bright_ti4"] <
    v11_events["mean_bright_ti4"]
).sum()

ti5_relationship_violations = (
    v11_events["max_bright_ti5"] <
    v11_events["mean_bright_ti5"]
).sum()

negative_range_violations = {
    column: (
        v11_events[column] < 0
    ).sum()
    for column in [
        "frp_range",
        "ti4_range",
        "ti5_range"
    ]
}

print("\nTHERMAL CONSISTENCY")
print("=" * 60)

print(
    "max_frp < mean_frp:",
    frp_relationship_violations
)

print(
    "max_bright_ti4 < mean_bright_ti4:",
    ti4_relationship_violations
)

print(
    "max_bright_ti5 < mean_bright_ti5:",
    ti5_relationship_violations
)

print("\nNegative ranges:")
print(negative_range_violations)

THERMAL FEATURE QC


,feature,minimum,maximum,mean,median,negative_count,zero_count,infinite_count
0,mean_frp,0.1100,21.3700,1.3318,1.0400,0,0,0
1,max_frp,0.1100,28.6300,1.4832,1.1100,0,0,0
2,std_frp,0.0000,10.9963,0.1195,0.0000,0,3717,0
3,mean_bright_ti4,295.0100,367.0000,304.9777,302.5300,0,0,0
4,max_bright_ti4,295.0100,367.0000,306.4857,303.3800,0,0,0
5,std_bright_ti4,0.0000,40.3192,1.2639,0.0000,0,3645,0
6,mean_bright_ti5,255.3000,298.1600,283.5181,284.7200,0,0,0
7,max_bright_ti5,255.3000,300.1800,283.8881,285.1500,0,0,0
8,std_bright_ti5,0.0000,16.1927,0.3570,0.0000,0,3647,0
9,frp_range,0.0000,14.7433,0.1513,0.0000,0,3717,0



THERMAL CONSISTENCY
max_frp < mean_frp: 0
max_bright_ti4 < mean_bright_ti4: 0
max_bright_ti5 < mean_bright_ti5: 0

Negative ranges:
{'frp_range': np.int64(0), 'ti4_range': np.int64(0), 'ti5_range': np.int64(0)}


In [11]:
# ============================================================
# CELL 9 — TEMPORAL FEATURE QC
# ============================================================

temporal_feature_columns = [
    "active_days",
    "duration_days",
    "activity_frequency"
]

temporal_qc_report = (
    v11_events[temporal_feature_columns]
    .describe()
    .T
)

print("TEMPORAL FEATURE DISTRIBUTIONS")
print("=" * 60)

display(
    temporal_qc_report.round(4)
)

# ------------------------------------------------------------
# Temporal consistency checks
# ------------------------------------------------------------

active_days_greater_than_duration = (
    v11_events["active_days"] >
    v11_events["duration_days"]
)

invalid_active_days_count = (
    (v11_events["active_days"] <= 0)
).sum()

invalid_duration_days_count = (
    (v11_events["duration_days"] <= 0)
).sum()

negative_activity_frequency_count = (
    v11_events["activity_frequency"] < 0
).sum()

activity_frequency_above_one_count = (
    v11_events["activity_frequency"] > 1
).sum()

print("\nTEMPORAL CONSISTENCY")
print("=" * 60)

print(
    "active_days > duration_days:",
    active_days_greater_than_duration.sum()
)

print(
    "active_days <= 0:",
    invalid_active_days_count
)

print(
    "duration_days <= 0:",
    invalid_duration_days_count
)

print(
    "activity_frequency < 0:",
    negative_activity_frequency_count
)

print(
    "activity_frequency > 1:",
    activity_frequency_above_one_count
)

TEMPORAL FEATURE DISTRIBUTIONS


,count,mean,std,min,25%,50%,75%,max
active_days,4893.0000,1.7370,2.8661,1.0000,1.0000,1.0000,1.0000,31.0000
duration_days,4893.0000,1.9787,3.5973,1.0000,1.0000,1.0000,1.0000,31.0000
activity_frequency,4893.0000,0.9716,0.0947,0.4286,1.0000,1.0000,1.0000,1.0000



TEMPORAL CONSISTENCY
active_days > duration_days: 0
active_days <= 0: 0
duration_days <= 0: 0
activity_frequency < 0: 0
activity_frequency > 1: 0


In [12]:
# ============================================================
# CELL 10 — DETECTION AND EVENT-SIZE QC
# ============================================================

event_size_columns = [
    "detection_count",
    "daily_object_count",
    "detections_per_active_day"
]

event_size_report = (
    v11_events[event_size_columns]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

print("EVENT SIZE DISTRIBUTION")
print("=" * 60)

display(
    event_size_report.round(4)
)

# ------------------------------------------------------------
# Internal relationships
# ------------------------------------------------------------

detection_less_than_daily_object_mask = (
    v11_events["detection_count"] <
    v11_events["daily_object_count"]
)

invalid_detection_count = (
    v11_events["detection_count"] <= 0
).sum()

invalid_daily_object_count = (
    v11_events["daily_object_count"] <= 0
).sum()

invalid_detection_density_count = (
    v11_events["detections_per_active_day"] <= 0
).sum()

print("\nEVENT SIZE CONSISTENCY")
print("=" * 60)

print(
    "detection_count < daily_object_count:",
    detection_less_than_daily_object_mask.sum()
)

print(
    "detection_count <= 0:",
    invalid_detection_count
)

print(
    "daily_object_count <= 0:",
    invalid_daily_object_count
)

print(
    "detections_per_active_day <= 0:",
    invalid_detection_density_count
)

EVENT SIZE DISTRIBUTION


,count,mean,std,min,50%,75%,90%,95%,99%,max
detection_count,4893.0000,2.7813,13.1317,1.0000,1.0000,2.0000,3.0000,6.4000,38.0800,694.0000
daily_object_count,4893.0000,2.3867,10.1482,1.0000,1.0000,1.0000,3.0000,6.0000,32.0000,515.0000
detections_per_active_day,4893.0000,1.1898,0.6436,1.0000,1.0000,1.0000,2.0000,2.0000,3.1718,25.7037



EVENT SIZE CONSISTENCY
detection_count < daily_object_count: 0
detection_count <= 0: 0
daily_object_count <= 0: 0
detections_per_active_day <= 0: 0


In [13]:
# ============================================================
# CELL 11 — SPATIAL / EVENT GEOMETRY QC
# ============================================================

spatial_feature_columns = [
    "spatial_diameter_km"
]

spatial_report = (
    v11_events[spatial_feature_columns]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

print("SPATIAL DIAMETER DISTRIBUTION")
print("=" * 60)

display(
    spatial_report.round(4)
)

negative_spatial_diameter_count = (
    v11_events["spatial_diameter_km"] < 0
).sum()

zero_spatial_diameter_count = (
    v11_events["spatial_diameter_km"] == 0
).sum()

spatial_diameter_thresholds = [
    0.375,
    0.500,
    1.000,
    1.500,
    2.000,
    3.000
]

spatial_threshold_report = pd.DataFrame({
    "threshold_km": spatial_diameter_thresholds,
    "event_count": [
        (
            v11_events["spatial_diameter_km"] >
            threshold
        ).sum()
        for threshold in spatial_diameter_thresholds
    ]
})

spatial_threshold_report["percentage"] = (
    spatial_threshold_report["event_count"]
    / len(v11_events)
    * 100
)

print("\nSPATIAL EXPANSION")
display(
    spatial_threshold_report.round(4)
)

print(
    "\nNegative spatial diameters:",
    negative_spatial_diameter_count
)

print(
    "Zero spatial diameters:",
    zero_spatial_diameter_count
)

SPATIAL DIAMETER DISTRIBUTION


,count,mean,std,min,50%,75%,90%,95%,99%,max
spatial_diameter_km,4893.0000,0.1008,0.2457,0.0000,0.0000,0.0600,0.3637,0.4988,1.0959,3.7180



SPATIAL EXPANSION


,threshold_km,event_count,percentage
0,0.3750,402,8.2158
1,0.5000,245,5.0072
2,1.0000,65,1.3284
3,1.5000,28,0.5722
4,2.0000,13,0.2657
5,3.0000,1,0.0204



Negative spatial diameters: 0
Zero spatial diameters: 3645


In [14]:
# ============================================================
# CELL 12 — INTERNAL CONSISTENCY AUDIT
# ============================================================

consistency_checks = {}

# ------------------------------------------------------------
# Temporal
# ------------------------------------------------------------

consistency_checks[
    "active_days_positive"
] = (
    v11_events["active_days"] > 0
)

consistency_checks[
    "duration_days_positive"
] = (
    v11_events["duration_days"] > 0
)

consistency_checks[
    "active_days_not_greater_than_duration"
] = (
    v11_events["active_days"] <=
    v11_events["duration_days"]
)

# ------------------------------------------------------------
# Detection counts
# ------------------------------------------------------------

consistency_checks[
    "detection_count_positive"
] = (
    v11_events["detection_count"] > 0
)

consistency_checks[
    "daily_object_count_positive"
] = (
    v11_events["daily_object_count"] > 0
)

consistency_checks[
    "daily_objects_not_greater_than_detections"
] = (
    v11_events["daily_object_count"] <=
    v11_events["detection_count"]
)

# ------------------------------------------------------------
# Thermal relationships
# ------------------------------------------------------------

consistency_checks[
    "max_frp_not_less_than_mean"
] = (
    v11_events["max_frp"] >=
    v11_events["mean_frp"]
)

consistency_checks[
    "max_ti4_not_less_than_mean"
] = (
    v11_events["max_bright_ti4"] >=
    v11_events["mean_bright_ti4"]
)

consistency_checks[
    "max_ti5_not_less_than_mean"
] = (
    v11_events["max_bright_ti5"] >=
    v11_events["mean_bright_ti5"]
)

# ------------------------------------------------------------
# Range features
# ------------------------------------------------------------

for range_column in [
    "frp_range",
    "ti4_range",
    "ti5_range"
]:
    consistency_checks[
        f"{range_column}_non_negative"
    ] = (
        v11_events[range_column] >= 0
    )

# ------------------------------------------------------------
# Spatial
# ------------------------------------------------------------

consistency_checks[
    "spatial_diameter_non_negative"
] = (
    v11_events["spatial_diameter_km"] >= 0
)

# ------------------------------------------------------------
# Calculate report
# ------------------------------------------------------------

consistency_records = []

for check_name, check_result in consistency_checks.items():

    failed_count = (~check_result).sum()

    consistency_records.append({
        "check": check_name,
        "passed_events": check_result.sum(),
        "failed_events": failed_count,
        "status": (
            "PASS"
            if failed_count == 0
            else "FAIL"
        )
    })

internal_consistency_report = pd.DataFrame(
    consistency_records
)

print("INTERNAL CONSISTENCY AUDIT")
print("=" * 60)

display(internal_consistency_report)

INTERNAL CONSISTENCY AUDIT


,check,passed_events,failed_events,status
0,active_days_positive,4893,0,PASS
1,duration_days_positive,4893,0,PASS
2,active_days_not_greater_than_duration,4893,0,PASS
3,detection_count_positive,4893,0,PASS
4,daily_object_count_positive,4893,0,PASS
5,daily_objects_not_greater_than_detections,4893,0,PASS
6,max_frp_not_less_than_mean,4893,0,PASS
7,max_ti4_not_less_than_mean,4893,0,PASS
8,max_ti5_not_less_than_mean,4893,0,PASS
9,frp_range_non_negative,4893,0,PASS


In [15]:
# ============================================================
# CELL 13 — DISTRIBUTION AND EXTREME-VALUE ANALYSIS
# ============================================================

distribution_columns = [
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km",
    "detections_per_active_day"
]

distribution_records = []

for column in distribution_columns:

    values = v11_events[column]

    distribution_records.append({
        "feature": column,
        "min": values.min(),
        "p01": values.quantile(0.01),
        "p05": values.quantile(0.05),
        "median": values.median(),
        "p95": values.quantile(0.95),
        "p99": values.quantile(0.99),
        "max": values.max(),
        "mean": values.mean(),
        "std": values.std()
    })

distribution_report = pd.DataFrame(
    distribution_records
)

print("FEATURE DISTRIBUTION REPORT")
print("=" * 60)

display(
    distribution_report.round(4)
)

FEATURE DISTRIBUTION REPORT


,feature,min,p01,p05,median,p95,p99,max,mean,std
0,mean_frp,0.1100,0.2700,0.4000,1.0400,3.1680,6.0554,21.3700,1.3318,1.2048
1,max_frp,0.1100,0.2700,0.4100,1.1100,3.6800,7.6224,28.6300,1.4832,1.4510
2,mean_bright_ti4,295.0100,295.1900,295.8460,302.5300,324.1893,337.8008,367.0000,304.9777,8.9031
3,max_bright_ti4,295.0100,295.1900,295.8600,303.3800,328.4140,343.0840,367.0000,306.4857,10.4194
4,active_days,1.0000,1.0000,1.0000,1.0000,5.0000,17.0000,31.0000,1.7370,2.8661
5,duration_days,1.0000,1.0000,1.0000,1.0000,7.0000,21.0000,31.0000,1.9787,3.5973
6,detection_count,1.0000,1.0000,1.0000,1.0000,6.4000,38.0800,694.0000,2.7813,13.1317
7,daily_object_count,1.0000,1.0000,1.0000,1.0000,6.0000,32.0000,515.0000,2.3867,10.1482
8,spatial_diameter_km,0.0000,0.0000,0.0000,0.0000,0.4988,1.0959,3.7180,0.1008,0.2457
9,detections_per_active_day,1.0000,1.0000,1.0000,1.0000,2.0000,3.1718,25.7037,1.1898,0.6436


In [16]:
# ============================================================
# CELL 14 — EXTREME EVENT INSPECTION
# ============================================================

extreme_event_columns = [
    "event_id",
    "centroid_lat",
    "centroid_lon",
    "start_date",
    "end_date",
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km",
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4"
]

# ------------------------------------------------------------
# Top events by different dimensions
# ------------------------------------------------------------

print("TOP 20 EVENTS — MAX FRP")
display(
    v11_events
    .nlargest(20, "max_frp")[
        extreme_event_columns
    ]
)

print("\nTOP 20 EVENTS — MAX TI4")
display(
    v11_events
    .nlargest(20, "max_bright_ti4")[
        extreme_event_columns
    ]
)

print("\nTOP 20 EVENTS — ACTIVE DAYS")
display(
    v11_events
    .nlargest(20, "active_days")[
        extreme_event_columns
    ]
)

print("\nTOP 20 EVENTS — SPATIAL DIAMETER")
display(
    v11_events
    .nlargest(20, "spatial_diameter_km")[
        extreme_event_columns
    ]
)

print("\nTOP 20 EVENTS — DETECTION COUNT")
display(
    v11_events
    .nlargest(20, "detection_count")[
        extreme_event_columns
    ]
)

TOP 20 EVENTS — MAX FRP


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
4016,4016,30.9375,75.6004,2024-01-26,2024-01-27,2,2,6,4,0.7260,13.8867,28.6300,327.6617,367.0000
867,867,22.0679,88.1212,2024-01-06,2024-01-06,1,1,1,1,0.0000,21.3700,21.3700,298.1000,298.1000
893,893,22.0644,88.1207,2024-01-06,2024-01-06,1,1,1,1,0.0000,21.3700,21.3700,344.2300,344.2300
3038,3038,33.0399,74.9502,2024-01-19,2024-01-19,1,1,1,1,0.0000,17.5400,17.5400,342.7500,342.7500
3039,3039,33.0406,74.9450,2024-01-19,2024-01-19,1,1,1,1,0.0000,17.5400,17.5400,327.7400,327.7400
48,48,21.1057,72.6407,2024-01-01,2024-01-31,31,31,205,157,2.6092,3.2355,15.6900,317.6236,354.2000
416,416,22.5106,88.3325,2024-01-03,2024-01-03,1,1,2,1,0.1921,10.6700,15.0100,361.0200,367.0000
4039,4039,30.9434,75.6022,2024-01-26,2024-01-27,2,2,3,2,0.2407,5.6233,14.8400,327.5400,367.0000
3069,3069,24.0698,69.5833,2024-01-20,2024-01-20,1,1,4,3,0.4891,7.3875,13.7700,333.3000,367.0000
3192,3192,24.0692,69.5759,2024-01-20,2024-01-20,1,1,1,1,0.0000,13.7700,13.7700,304.7400,304.7400



TOP 20 EVENTS — MAX TI4


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
416,416,22.5106,88.3325,2024-01-03,2024-01-03,1,1,2,1,0.1921,10.6700,15.0100,361.0200,367.0000
543,543,26.8375,95.2460,2024-01-04,2024-01-04,1,1,3,2,0.3882,5.5533,7.6800,332.3633,367.0000
624,624,26.8431,95.2459,2024-01-04,2024-01-04,1,1,1,1,0.0000,7.6800,7.6800,367.0000,367.0000
1686,1686,13.1520,80.1974,2024-01-12,2024-01-12,1,1,3,1,0.3529,6.1533,11.6100,334.0033,367.0000
1881,1881,17.6007,83.1822,2024-01-13,2024-01-14,2,2,5,4,0.6742,5.9200,8.4900,328.6020,367.0000
3069,3069,24.0698,69.5833,2024-01-20,2024-01-20,1,1,4,3,0.4891,7.3875,13.7700,333.3000,367.0000
4016,4016,30.9375,75.6004,2024-01-26,2024-01-27,2,2,6,4,0.7260,13.8867,28.6300,327.6617,367.0000
4039,4039,30.9434,75.6022,2024-01-26,2024-01-27,2,2,3,2,0.2407,5.6233,14.8400,327.5400,367.0000
18,18,22.0420,83.7352,2024-01-01,2024-01-31,27,31,113,96,1.4593,2.6027,6.3400,314.2127,356.5200
3460,3460,15.7168,78.6502,2024-01-22,2024-01-22,1,1,1,1,0.0000,3.6700,3.6700,354.9700,354.9700



TOP 20 EVENTS — ACTIVE DAYS


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
48,48,21.1057,72.6407,2024-01-01,2024-01-31,31,31,205,157,2.6092,3.2355,15.6900,317.6236,354.2000
40,40,21.4870,81.7687,2024-01-01,2024-01-31,29,31,46,38,0.6968,1.5280,2.6700,306.6661,318.0600
43,43,18.6850,73.0361,2024-01-01,2024-01-31,28,31,94,81,1.8755,1.8650,4.8200,308.7644,340.4000
0,0,23.1695,82.3411,2024-01-01,2024-01-31,27,31,85,75,1.5451,1.6354,4.9700,307.7935,337.4800
2,2,22.0539,88.1241,2024-01-01,2024-01-31,27,31,85,64,1.0219,1.7355,3.9000,309.4428,330.2000
17,17,23.7697,86.3942,2024-01-01,2024-01-31,27,31,694,515,3.7180,1.8498,5.1300,308.3440,343.5600
18,18,22.0420,83.7352,2024-01-01,2024-01-31,27,31,113,96,1.4593,2.6027,6.3400,314.2127,356.5200
36,36,15.1715,76.3785,2024-01-01,2024-01-31,27,31,56,46,1.1537,1.4175,3.1500,305.1409,317.5700
100,100,23.7192,86.4440,2024-01-01,2024-01-31,26,31,96,78,1.4095,1.8621,4.8000,305.9434,319.7000
127,127,15.1759,76.6577,2024-01-01,2024-01-31,26,31,86,75,2.5376,2.0965,5.0800,312.2988,344.2900



TOP 20 EVENTS — SPATIAL DIAMETER


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
17,17,23.7697,86.3942,2024-01-01,2024-01-31,27,31,694,515,3.7180,1.8498,5.1300,308.3440,343.5600
48,48,21.1057,72.6407,2024-01-01,2024-01-31,31,31,205,157,2.6092,3.2355,15.6900,317.6236,354.2000
127,127,15.1759,76.6577,2024-01-01,2024-01-31,26,31,86,75,2.5376,2.0965,5.0800,312.2988,344.2900
80,80,21.7628,84.0212,2024-01-01,2024-01-21,19,21,125,100,2.4513,1.9406,5.0100,308.0627,331.2400
135,135,20.9664,85.1727,2024-01-01,2024-01-17,16,17,177,135,2.4327,1.9717,4.6700,310.2730,342.1200
19,19,20.7907,85.2585,2024-01-01,2024-01-18,17,18,97,74,2.2939,2.3676,8.9800,310.5990,346.7600
25,25,23.5561,87.2407,2024-01-01,2024-01-14,14,14,63,53,2.2673,1.5333,4.0400,306.3963,335.2200
91,91,23.6916,87.1168,2024-01-01,2024-01-15,15,15,99,81,2.2545,1.6256,3.3000,302.4336,317.0300
2899,2899,23.6919,87.1179,2024-01-19,2024-01-31,9,13,49,42,2.2334,1.4084,4.2100,301.9302,313.1100
193,193,23.7776,86.2056,2024-01-01,2024-01-31,26,31,159,130,2.1293,2.0575,7.3400,308.5401,348.6000



TOP 20 EVENTS — DETECTION COUNT


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
17,17,23.7697,86.3942,2024-01-01,2024-01-31,27,31,694,515,3.7180,1.8498,5.1300,308.3440,343.5600
48,48,21.1057,72.6407,2024-01-01,2024-01-31,31,31,205,157,2.6092,3.2355,15.6900,317.6236,354.2000
135,135,20.9664,85.1727,2024-01-01,2024-01-17,16,17,177,135,2.4327,1.9717,4.6700,310.2730,342.1200
193,193,23.7776,86.2056,2024-01-01,2024-01-31,26,31,159,130,2.1293,2.0575,7.3400,308.5401,348.6000
80,80,21.7628,84.0212,2024-01-01,2024-01-21,19,21,125,100,2.4513,1.9406,5.0100,308.0627,331.2400
140,140,23.6848,86.3922,2024-01-01,2024-01-31,26,31,115,90,1.8995,1.8863,5.1500,308.2620,332.5100
18,18,22.0420,83.7352,2024-01-01,2024-01-31,27,31,113,96,1.4593,2.6027,6.3400,314.2127,356.5200
79,79,22.7907,86.2033,2024-01-01,2024-01-16,16,16,111,92,2.0885,1.6532,5.2100,306.0396,333.3400
10,10,22.3279,82.6559,2024-01-01,2024-01-26,22,26,104,88,1.8281,1.5051,3.5000,306.1647,324.7000
91,91,23.6916,87.1168,2024-01-01,2024-01-15,15,15,99,81,2.2545,1.6256,3.3000,302.4336,317.0300


In [17]:
# ============================================================
# CELL 15 — EXTREME COMBINATION CHECK
# ============================================================

extreme_event_mask = (
    (v11_events["active_days"] >= 15) |
    (v11_events["max_frp"] >= 10) |
    (v11_events["max_bright_ti4"] >= 350) |
    (v11_events["spatial_diameter_km"] >= 2.0) |
    (v11_events["detection_count"] >= 100)
)

extreme_events = (
    v11_events.loc[
        extreme_event_mask,
        extreme_event_columns
    ]
    .sort_values(
        [
            "active_days",
            "max_frp",
            "spatial_diameter_km"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Extreme-event candidate count:", len(extreme_events))

display(extreme_events)

Extreme-event candidate count: 104


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4
0,48,21.1057,72.6407,2024-01-01,2024-01-31,31,31,205,157,2.6092,3.2355,15.6900,317.6236,354.2000
1,40,21.4870,81.7687,2024-01-01,2024-01-31,29,31,46,38,0.6968,1.5280,2.6700,306.6661,318.0600
2,43,18.6850,73.0361,2024-01-01,2024-01-31,28,31,94,81,1.8755,1.8650,4.8200,308.7644,340.4000
3,18,22.0420,83.7352,2024-01-01,2024-01-31,27,31,113,96,1.4593,2.6027,6.3400,314.2127,356.5200
4,17,23.7697,86.3942,2024-01-01,2024-01-31,27,31,694,515,3.7180,1.8498,5.1300,308.3440,343.5600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,624,26.8431,95.2459,2024-01-04,2024-01-04,1,1,1,1,0.0000,7.6800,7.6800,367.0000,367.0000
100,3650,18.7099,73.2710,2024-01-24,2024-01-24,1,1,2,1,0.3729,4.9300,6.9400,340.9850,353.6800
101,2288,31.5789,77.8924,2024-01-14,2024-01-14,1,1,1,1,0.0000,4.3800,4.3800,353.6100,353.6100
102,2902,26.7630,90.9338,2024-01-19,2024-01-19,1,1,2,1,0.1121,3.2100,3.7100,335.5300,350.2000


In [18]:
# ============================================================
# CELL 16 — COORDINATE DUPLICATION CHECK
# ============================================================

coordinate_duplicate_mask = (
    v11_events
    .duplicated(
        subset=[
            "centroid_lat",
            "centroid_lon"
        ],
        keep=False
    )
)

coordinate_duplicate_events = (
    v11_events.loc[
        coordinate_duplicate_mask,
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "start_date",
            "end_date",
            "active_days",
            "detection_count"
        ]
    ]
    .sort_values(
        [
            "centroid_lat",
            "centroid_lon",
            "start_date"
        ]
    )
)

print(
    "Events sharing identical centroid coordinates:",
    coordinate_duplicate_mask.sum()
)

display(coordinate_duplicate_events.head(100))

Events sharing identical centroid coordinates: 0


,event_id,centroid_lat,centroid_lon,start_date,end_date,active_days,detection_count


In [19]:
# ============================================================
# CELL 17 — INDIA GEOGRAPHIC PLAUSIBILITY CHECK
# ============================================================

# Broad bounding box covering India's geographic extent.
# This is intentionally wider than the exact national boundary.

INDIA_MIN_LATITUDE = 5.0
INDIA_MAX_LATITUDE = 38.5

INDIA_MIN_LONGITUDE = 67.0
INDIA_MAX_LONGITUDE = 99.0

outside_india_latitude_mask = (
    (v11_events["centroid_lat"] < INDIA_MIN_LATITUDE) |
    (v11_events["centroid_lat"] > INDIA_MAX_LATITUDE)
)

outside_india_longitude_mask = (
    (v11_events["centroid_lon"] < INDIA_MIN_LONGITUDE) |
    (v11_events["centroid_lon"] > INDIA_MAX_LONGITUDE)
)

outside_india_bbox_mask = (
    outside_india_latitude_mask |
    outside_india_longitude_mask
)

india_plausibility_report = pd.DataFrame({
    "check": [
        "Outside broad India latitude range",
        "Outside broad India longitude range",
        "Outside broad India bounding box"
    ],
    "event_count": [
        outside_india_latitude_mask.sum(),
        outside_india_longitude_mask.sum(),
        outside_india_bbox_mask.sum()
    ]
})

print("INDIA GEOGRAPHIC PLAUSIBILITY")
print("=" * 60)

display(india_plausibility_report)

if outside_india_bbox_mask.sum() > 0:
    print("\nEvents requiring geographic inspection:")
    display(
        v11_events.loc[
            outside_india_bbox_mask,
            extreme_event_columns
        ]
    )
else:
    print(
        "STATUS: PASS — All event centroids fall "
        "within the broad India bounding box."
    )

INDIA GEOGRAPHIC PLAUSIBILITY


,check,event_count
0,Outside broad India latitude range,0
1,Outside broad India longitude range,0
2,Outside broad India bounding box,0


STATUS: PASS — All event centroids fall within the broad India bounding box.


In [20]:
# ============================================================
# CELL 18 — FINAL QC STATUS SUMMARY
# ============================================================

qc_status_records = []

# ------------------------------------------------------------
# Dataset size
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Dataset size",
    "check": "Expected event count",
    "observed": len(v11_events),
    "threshold": EXPECTED_EVENT_COUNT,
    "status": (
        "PASS"
        if len(v11_events) == EXPECTED_EVENT_COUNT
        else "WARNING"
    )
})

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Missing values",
    "check": "Total missing values",
    "observed": total_missing_values,
    "threshold": 0,
    "status": (
        "PASS"
        if total_missing_values == 0
        else "FAIL"
    )
})

# ------------------------------------------------------------
# Event IDs
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Event IDs",
    "check": "Duplicate event IDs",
    "observed": duplicate_event_id_count,
    "threshold": 0,
    "status": (
        "PASS"
        if duplicate_event_id_count == 0
        else "FAIL"
    )
})

# ------------------------------------------------------------
# Coordinates
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Coordinates",
    "check": "Invalid latitude",
    "observed": invalid_latitude_count,
    "threshold": 0,
    "status": (
        "PASS"
        if invalid_latitude_count == 0
        else "FAIL"
    )
})

qc_status_records.append({
    "qc_category": "Coordinates",
    "check": "Invalid longitude",
    "observed": invalid_longitude_count,
    "threshold": 0,
    "status": (
        "PASS"
        if invalid_longitude_count == 0
        else "FAIL"
    )
})

# ------------------------------------------------------------
# Dates
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Dates",
    "check": "End date before start date",
    "observed": end_before_start_count,
    "threshold": 0,
    "status": (
        "PASS"
        if end_before_start_count == 0
        else "FAIL"
    )
})

# ------------------------------------------------------------
# Internal consistency
# ------------------------------------------------------------

for _, row in internal_consistency_report.iterrows():

    qc_status_records.append({
        "qc_category": "Internal consistency",
        "check": row["check"],
        "observed": row["failed_events"],
        "threshold": 0,
        "status": row["status"]
    })

# ------------------------------------------------------------
# Spatial
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Spatial",
    "check": "Negative spatial diameter",
    "observed": negative_spatial_diameter_count,
    "threshold": 0,
    "status": (
        "PASS"
        if negative_spatial_diameter_count == 0
        else "FAIL"
    )
})

# ------------------------------------------------------------
# India plausibility
# ------------------------------------------------------------

qc_status_records.append({
    "qc_category": "Geographic scope",
    "check": "Outside broad India bounding box",
    "observed": outside_india_bbox_mask.sum(),
    "threshold": 0,
    "status": (
        "PASS"
        if outside_india_bbox_mask.sum() == 0
        else "WARNING"
    )
})

final_qc_status_report = pd.DataFrame(
    qc_status_records
)

print("FINAL QC STATUS")
print("=" * 60)

display(final_qc_status_report)

print("\nSTATUS COUNTS")
display(
    final_qc_status_report["status"]
    .value_counts()
    .to_frame("count")
)

FINAL QC STATUS


,qc_category,check,observed,threshold,status
0,Dataset size,Expected event count,4893,4893,PASS
1,Missing values,Total missing values,0,0,PASS
2,Event IDs,Duplicate event IDs,0,0,PASS
3,Coordinates,Invalid latitude,0,0,PASS
4,Coordinates,Invalid longitude,0,0,PASS
5,Dates,End date before start date,0,0,PASS
6,Internal consistency,active_days_positive,0,0,PASS
7,Internal consistency,duration_days_positive,0,0,PASS
8,Internal consistency,active_days_not_greater_than_duration,0,0,PASS
9,Internal consistency,detection_count_positive,0,0,PASS



STATUS COUNTS


,count
status,
PASS,21


In [21]:
# ============================================================
# CELL 19 — SAVE QC REPORTS
# ============================================================

missing_value_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "missing_value_report.csv"
    ),
    index=False
)

event_id_integrity_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "event_id_integrity_report.csv"
    ),
    index=False
)

thermal_qc_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "thermal_qc_report.csv"
    ),
    index=False
)

temporal_qc_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "temporal_qc_report.csv"
    ),
    index=False
)

event_size_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "event_size_report.csv"
    ),
    index=False
)

spatial_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "spatial_qc_report.csv"
    ),
    index=False
)

internal_consistency_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "internal_consistency_report.csv"
    ),
    index=False
)

distribution_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "feature_distribution_report.csv"
    ),
    index=False
)

spatial_threshold_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "spatial_expansion_report.csv"
    ),
    index=False
)

extreme_events.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "extreme_events_for_review.csv"
    ),
    index=False
)

final_qc_status_report.to_csv(
    os.path.join(
        QC_OUTPUT_DIRECTORY,
        "final_qc_status_report.csv"
    ),
    index=False
)

print("QC reports saved to:")
print(QC_OUTPUT_DIRECTORY)

QC reports saved to:
dataset_qc


In [60]:
# ============================================================
# CELL 20 — LOAD V11 DETECTION-TO-EVENT MAPPING
# ============================================================

DETECTION_EVENT_MAPPING_FILE = (
    "viirs_v11_detection_event_mapping.csv"
)

if not os.path.exists(DETECTION_EVENT_MAPPING_FILE):
    raise FileNotFoundError(
        f"V11 mapping not found: "
        f"{DETECTION_EVENT_MAPPING_FILE}"
    )

detection_event_mapping = pd.read_csv(
    DETECTION_EVENT_MAPPING_FILE,
    parse_dates=["acq_date"]
)

print("Detection-to-event mapping loaded.")
print()
print("Shape:", detection_event_mapping.shape)
print("Rows :", len(detection_event_mapping))
print(
    "Columns:",
    detection_event_mapping.columns.tolist()
)

Detection-to-event mapping loaded.

Shape: (13609, 6)
Rows : 13609
Columns: ['detection_index', 'daily_object_id', 'event_id', 'acq_date', 'latitude', 'longitude']


In [61]:
# ============================================================
# CELL 21 — V11 MAPPING SCHEMA CHECK
# ============================================================

required_mapping_columns = [
    "detection_index",
    "daily_object_id",
    "event_id",
    "acq_date"
]

missing_mapping_columns = [
    col for col in required_mapping_columns
    if col not in detection_event_mapping.columns
]

mapping_schema_status = (
    "PASS"
    if len(missing_mapping_columns) == 0
    else "FAIL"
)

mapping_schema_report = pd.DataFrame({
    "check": ["Required mapping columns"],
    "missing_columns": [
        ", ".join(missing_mapping_columns)
        if missing_mapping_columns
        else "None"
    ],
    "status": [mapping_schema_status]
})

display(mapping_schema_report)

,check,missing_columns,status
0,Required mapping columns,None,PASS


In [62]:
# ============================================================
# CELL 22 — DETECTION COUNT RECONCILIATION
# ============================================================

mapping_detection_count = (
    detection_event_mapping["detection_index"].nunique()
)

v11_detection_count = len(jan_night) if "jan_night" in globals() else 13609

detection_count_reconciliation = pd.DataFrame({
    "quantity": ["VIIRS detections"],
    "expected": [v11_detection_count],
    "observed": [mapping_detection_count],
    "difference": [
        mapping_detection_count - v11_detection_count
    ]
})

detection_count_reconciliation["status"] = np.where(
    detection_count_reconciliation["difference"] == 0,
    "PASS",
    "FAIL"
)

display(detection_count_reconciliation)

,quantity,expected,observed,difference,status
0,VIIRS detections,13609,13609,0,PASS


In [63]:
# ============================================================
# CELL 23 — EVENT ID RECONCILIATION
# ============================================================

dataset_event_ids = set(
    v11_events["event_id"]
)

mapping_event_ids = set(
    detection_event_mapping["event_id"]
)

missing_from_mapping = (
    dataset_event_ids - mapping_event_ids
)

extra_in_mapping = (
    mapping_event_ids - dataset_event_ids
)

event_id_reconciliation = pd.DataFrame({
    "check": [
        "V11 events missing from mapping",
        "Extra events in mapping",
        "V11 event count",
        "Mapping event count"
    ],
    "value": [
        len(missing_from_mapping),
        len(extra_in_mapping),
        len(dataset_event_ids),
        len(mapping_event_ids)
    ],
    "status": [
        "PASS" if len(missing_from_mapping) == 0 else "FAIL",
        "PASS" if len(extra_in_mapping) == 0 else "FAIL",
        "PASS" if len(dataset_event_ids) == 4893 else "FAIL",
        "PASS" if len(mapping_event_ids) == 4893 else "FAIL"
    ]
})

display(event_id_reconciliation)

,check,value,status
0,V11 events missing from mapping,0,PASS
1,Extra events in mapping,0,PASS
2,V11 event count,4893,PASS
3,Mapping event count,4893,PASS


In [64]:
# ============================================================
# CELL 24 — DETECTION ASSIGNMENT UNIQUENESS
# ============================================================

duplicate_detection_assignments = (
    detection_event_mapping[
        detection_event_mapping["detection_index"].duplicated(
            keep=False
        )
    ]
    .sort_values("detection_index")
)

duplicate_detection_count = (
    duplicate_detection_assignments[
        "detection_index"
    ].nunique()
)

detection_assignment_report = pd.DataFrame({
    "check": [
        "Duplicate detection assignments"
    ],
    "count": [
        duplicate_detection_count
    ],
    "status": [
        "PASS"
        if duplicate_detection_count == 0
        else "FAIL"
    ]
})

display(detection_assignment_report)

if duplicate_detection_count > 0:
    display(duplicate_detection_assignments.head(20))

,check,count,status
0,Duplicate detection assignments,0,PASS


In [65]:
# ============================================================
# CELL 25 — DETECTION ASSIGNMENT COMPLETENESS
# ============================================================

if "jan_night" in globals():
    expected_detection_indices = set(
        jan_night.index
    )
else:
    expected_detection_indices = set(
        range(13609)
    )

mapped_detection_indices = set(
    detection_event_mapping["detection_index"]
)

missing_detection_indices = (
    expected_detection_indices -
    mapped_detection_indices
)

extra_detection_indices = (
    mapped_detection_indices -
    expected_detection_indices
)

assignment_completeness_report = pd.DataFrame({
    "check": [
        "Unassigned detections",
        "Unexpected detection indices"
    ],
    "count": [
        len(missing_detection_indices),
        len(extra_detection_indices)
    ],
    "status": [
        "PASS"
        if len(missing_detection_indices) == 0
        else "FAIL",
        "PASS"
        if len(extra_detection_indices) == 0
        else "FAIL"
    ]
})

display(assignment_completeness_report)

,check,count,status
0,Unassigned detections,0,PASS
1,Unexpected detection indices,0,PASS


In [66]:
# ============================================================
# CELL 26 — DAILY OBJECT → EVENT UNIQUENESS
# ============================================================

daily_object_event_counts = (
    detection_event_mapping
    .groupby("daily_object_id")["event_id"]
    .nunique()
)

multiple_event_daily_objects = (
    daily_object_event_counts[
        daily_object_event_counts > 1
    ]
)

daily_object_event_report = pd.DataFrame({
    "check": [
        "Unique daily objects",
        "Daily objects assigned to multiple events"
    ],
    "count": [
        detection_event_mapping["daily_object_id"].nunique(),
        len(multiple_event_daily_objects)
    ],
    "status": [
        "PASS"
        if detection_event_mapping[
            "daily_object_id"
        ].nunique() == 11678
        else "FAIL",

        "PASS"
        if len(multiple_event_daily_objects) == 0
        else "FAIL"
    ]
})

display(daily_object_event_report)

,check,count,status
0,Unique daily objects,11678,PASS
1,Daily objects assigned to multiple events,0,PASS


In [67]:
# ============================================================
# CELL 27 — DAILY OBJECT DATE CONSISTENCY
# ============================================================

daily_object_date_counts = (
    detection_event_mapping
    .groupby("daily_object_id")["acq_date"]
    .nunique()
)

multi_date_daily_objects = (
    daily_object_date_counts[
        daily_object_date_counts > 1
    ]
)

daily_object_date_report = pd.DataFrame({
    "check": [
        "Daily objects containing multiple dates"
    ],
    "count": [
        len(multi_date_daily_objects)
    ],
    "status": [
        "PASS"
        if len(multi_date_daily_objects) == 0
        else "FAIL"
    ]
})

display(daily_object_date_report)

,check,count,status
0,Daily objects containing multiple dates,0,PASS


In [68]:
# ============================================================
# CELL 28 — RECONSTRUCT V11 DAILY OBJECT GEOMETRY
# ============================================================

from sklearn.neighbors import BallTree

EARTH_RADIUS_KM = 6371.0088

required_geometry_columns = [
    "detection_index",
    "daily_object_id",
    "latitude",
    "longitude"
]

missing_geometry_columns = [
    col for col in required_geometry_columns
    if col not in detection_event_mapping.columns
]

if missing_geometry_columns:
    raise ValueError(
        "V11 mapping is missing required geometry columns: "
        + ", ".join(missing_geometry_columns)
    )

daily_object_geometry_records = []

for daily_object_id, group in detection_event_mapping.groupby(
    "daily_object_id"
):

    coords_rad = np.radians(
        group[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    if len(coords_rad) <= 1:

        diameter_km = 0.0

    else:

        tree = BallTree(
            coords_rad,
            metric="haversine"
        )

        max_distance_rad = 0.0

        for point in coords_rad:

            distances, _ = tree.query(
                point.reshape(1, -1),
                k=len(coords_rad)
            )

            max_distance_rad = max(
                max_distance_rad,
                distances.max()
            )

        diameter_km = (
            max_distance_rad *
            EARTH_RADIUS_KM
        )

    daily_object_geometry_records.append({
        "daily_object_id": daily_object_id,
        "detection_count": len(group),
        "spatial_diameter_km": diameter_km
    })

daily_object_geometry = pd.DataFrame(
    daily_object_geometry_records
)

print(
    "Daily objects reconstructed:",
    len(daily_object_geometry)
)

print(
    "Maximum diameter (km):",
    daily_object_geometry[
        "spatial_diameter_km"
    ].max()
)

display(
    daily_object_geometry.head()
)

Daily objects reconstructed: 11678
Maximum diameter (km): 0.37498388341558436


,daily_object_id,detection_count,spatial_diameter_km
0,0,2,0.3720
1,1,2,0.3679
2,2,2,0.3657
3,3,2,0.3496
4,4,2,0.3579


In [69]:
# ============================================================
# CELL 29 — V11 DAILY SPATIAL CONSTRAINT
# ============================================================

DAILY_OBJECT_MAX_DIAMETER_KM = 0.375

violating_daily_objects = (
    daily_object_geometry[
        daily_object_geometry[
            "spatial_diameter_km"
        ] > DAILY_OBJECT_MAX_DIAMETER_KM
    ]
    .copy()
)

daily_spatial_constraint_report = pd.DataFrame({
    "check": [
        "Total V11 daily objects",
        "Daily objects exceeding 375 m",
        "Maximum daily-object diameter (km)"
    ],
    "value": [
        len(daily_object_geometry),
        len(violating_daily_objects),
        daily_object_geometry[
            "spatial_diameter_km"
        ].max()
    ],
    "status": [
        "PASS"
        if len(daily_object_geometry) == 11678
        else "FAIL",

        "PASS"
        if len(violating_daily_objects) == 0
        else "FAIL",

        "PASS"
        if daily_object_geometry[
            "spatial_diameter_km"
        ].max() <= 0.375
        else "FAIL"
    ]
})

display(daily_spatial_constraint_report)

if len(violating_daily_objects) > 0:
    display(
        violating_daily_objects.sort_values(
            "spatial_diameter_km",
            ascending=False
        ).head(20)
    )

,check,value,status
0,Total V11 daily objects,11678.0000,PASS
1,Daily objects exceeding 375 m,0.0000,PASS
2,Maximum daily-object diameter (km),0.3750,PASS


In [70]:
# ============================================================
# CELL 30 — EVENT COUNT RECONCILIATION
# ============================================================

mapping_event_counts = (
    detection_event_mapping
    .groupby("event_id")
    .agg(
        reconstructed_detection_count=(
            "detection_index",
            "nunique"
        ),
        reconstructed_daily_object_count=(
            "daily_object_id",
            "nunique"
        )
    )
    .reset_index()
)

event_count_reconciliation = (
    v11_events[
        [
            "event_id",
            "detection_count",
            "daily_object_count"
        ]
    ]
    .merge(
        mapping_event_counts,
        on="event_id",
        how="outer",
        indicator=True
    )
)

event_count_reconciliation[
    "detection_count_match"
] = (
    event_count_reconciliation[
        "detection_count"
    ]
    ==
    event_count_reconciliation[
        "reconstructed_detection_count"
    ]
)

event_count_reconciliation[
    "daily_object_count_match"
] = (
    event_count_reconciliation[
        "daily_object_count"
    ]
    ==
    event_count_reconciliation[
        "reconstructed_daily_object_count"
    ]
)

display(
    event_count_reconciliation.head()
)

print(
    "Detection-count mismatches:",
    (
        ~event_count_reconciliation[
            "detection_count_match"
        ]
    ).sum()
)

print(
    "Daily-object-count mismatches:",
    (
        ~event_count_reconciliation[
            "daily_object_count_match"
        ]
    ).sum()
)

,event_id,detection_count,daily_object_count,reconstructed_detection_count,reconstructed_daily_object_count,_merge,detection_count_match,daily_object_count_match
0,0,85,75,85,75,both,True,True
1,1,3,2,3,2,both,True,True
2,2,85,64,85,64,both,True,True
3,3,3,2,3,2,both,True,True
4,4,2,1,2,1,both,True,True


Detection-count mismatches: 0
Daily-object-count mismatches: 0


In [71]:
# ============================================================
# CELL 31 — EVENT TEMPORAL RECONCILIATION
# ============================================================

mapping_temporal = (
    detection_event_mapping
    .groupby("event_id")
    .agg(
        reconstructed_start_date=(
            "acq_date",
            "min"
        ),
        reconstructed_end_date=(
            "acq_date",
            "max"
        ),
        reconstructed_active_days=(
            "acq_date",
            "nunique"
        )
    )
    .reset_index()
)

mapping_temporal[
    "reconstructed_start_date"
] = pd.to_datetime(
    mapping_temporal["reconstructed_start_date"]
)

mapping_temporal[
    "reconstructed_end_date"
] = pd.to_datetime(
    mapping_temporal["reconstructed_end_date"]
)

event_temporal_reconciliation = (
    v11_events[
        [
            "event_id",
            "start_date",
            "end_date",
            "active_days"
        ]
    ]
    .copy()
)

event_temporal_reconciliation[
    "start_date"
] = pd.to_datetime(
    event_temporal_reconciliation["start_date"]
)

event_temporal_reconciliation[
    "end_date"
] = pd.to_datetime(
    event_temporal_reconciliation["end_date"]
)

event_temporal_reconciliation = (
    event_temporal_reconciliation
    .merge(
        mapping_temporal,
        on="event_id",
        how="outer",
        indicator=True
    )
)

event_temporal_reconciliation[
    "start_date_match"
] = (
    event_temporal_reconciliation["start_date"]
    ==
    event_temporal_reconciliation[
        "reconstructed_start_date"
    ]
)

event_temporal_reconciliation[
    "end_date_match"
] = (
    event_temporal_reconciliation["end_date"]
    ==
    event_temporal_reconciliation[
        "reconstructed_end_date"
    ]
)

event_temporal_reconciliation[
    "active_days_match"
] = (
    event_temporal_reconciliation["active_days"]
    ==
    event_temporal_reconciliation[
        "reconstructed_active_days"
    ]
)

display(
    event_temporal_reconciliation.head()
)

print(
    "Start-date mismatches:",
    (
        ~event_temporal_reconciliation[
            "start_date_match"
        ]
    ).sum()
)

print(
    "End-date mismatches:",
    (
        ~event_temporal_reconciliation[
            "end_date_match"
        ]
    ).sum()
)

print(
    "Active-day mismatches:",
    (
        ~event_temporal_reconciliation[
            "active_days_match"
        ]
    ).sum()
)

,event_id,start_date,end_date,active_days,reconstructed_start_date,reconstructed_end_date,reconstructed_active_days,_merge,start_date_match,end_date_match,active_days_match
0,0,2024-01-01,2024-01-31,27,2024-01-01,2024-01-31,27,both,True,True,True
1,1,2024-01-01,2024-01-02,2,2024-01-01,2024-01-02,2,both,True,True,True
2,2,2024-01-01,2024-01-31,27,2024-01-01,2024-01-31,27,both,True,True,True
3,3,2024-01-01,2024-01-02,2,2024-01-01,2024-01-02,2,both,True,True,True
4,4,2024-01-01,2024-01-01,1,2024-01-01,2024-01-01,1,both,True,True,True


Start-date mismatches: 0
End-date mismatches: 0
Active-day mismatches: 0
